# Pipeline en Colab: descarga de PopResume + OCR con PaddleOCR

Ejecuta las celdas **en orden, de arriba hacia abajo** (botón ▶ a la izquierda de cada celda).
Antes de empezar:
1. `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar`.
2. En el panel izquierdo (icono 🔑 **Secretos**) crea:
   - `HF_TOKEN` = tu token de Hugging Face (el que empieza con `hf_`), con **acceso al notebook activado**.
   - `GH_TOKEN` = *solo si tu repo es privado*: un token de GitHub con permiso de lectura al repo.
3. Mantén esta pestaña abierta mientras corre el OCR.

## 1. Verificar que hay GPU

In [ ]:
!nvidia-smi

## 2. Conectar Google Drive (aquí se guardan los datos para que no se pierdan)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clonar tu repositorio de GitHub

In [ ]:
import os
from google.colab import userdata

GITHUB_USER = "patriciacastr"
REPO = "HW_04_202602_cv_experience_ocr_bert"
BRANCH = "main"   # si trabajas en una rama sin fusionar, pon aquí su nombre

try:
    gh = userdata.get("GH_TOKEN")
    url = f"https://{gh}@github.com/{GITHUB_USER}/{REPO}.git"   # repo privado
except Exception:
    url = f"https://github.com/{GITHUB_USER}/{REPO}.git"        # repo público

if not os.path.exists(f"/content/{REPO}"):
    !git clone -b {BRANCH} {url} /content/{REPO}
else:
    !cd /content/{REPO} && git pull origin {BRANCH}

%cd /content/{REPO}
!ls

## 4. Instalar PaddlePaddle (GPU) y las demás librerías

Tarda unos minutos. **Después de esta celda reinicia la sesión** (`Entorno de ejecución → Reiniciar sesión`) y vuelve a ejecutar solo las celdas 2 y 3 antes de la comprobación. No repitas la instalación.


In [ ]:
!pip install -q paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q paddleocr datasets huggingface_hub pandas tqdm pillow jiwer
# PyTorch choca con PaddlePaddle en Colab (NCCL) y no se necesita para el OCR: se elimina por completo.
!pip uninstall -y -q torch torchvision torchaudio torchao torchcodec torchdata torchsummary torchtune
!rm -rf /usr/local/lib/python3.13/dist-packages/torch /usr/local/lib/python3.13/dist-packages/torchvision /usr/local/lib/python3.13/dist-packages/torchaudio


In [ ]:
import paddle, paddleocr
print("Paddle:", paddle.__version__, "|", paddle.device.get_device())
print("PaddleOCR:", paddleocr.__version__)
# Ejecuta esta celda UNA sola vez; si falla, reinicia la sesión antes de repetir.


## 5. Iniciar sesión en Hugging Face (usa el secreto `HF_TOKEN`)

In [ ]:
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))

## 6. Guardar datos en Drive
Enlaza las carpetas `data/raw`, `data/ocr`, `data/processed` y `data/splits` con tu Drive.
Así, si Colab se desconecta, **nada se pierde** y el OCR puede continuar donde quedó.

In [ ]:
DRIVE = "/content/drive/MyDrive/cv_experience_ocr_bert_data"
for name in ["raw", "ocr", "processed", "splits"]:
    os.makedirs(f"{DRIVE}/{name}", exist_ok=True)
    target = f"/content/{REPO}/data/{name}"
    !rm -rf {target}
    !ln -s {DRIVE}/{name} {target}
!ls -la data/

## 7. PRUEBA pequeña (5 CVs por ocupación) — hazla antes de la descarga completa

In [ ]:
!python src/download_data.py --n-per-occ 5 --seed 42 --buffer-size 50

In [ ]:
!python src/ocr_extraction.py --limit 20

In [ ]:
import json, glob
f = sorted(glob.glob("data/ocr/*.json"))[0]
d = json.load(open(f))
print("Archivo:", f, "| líneas:", d["n_lines"], "| confianza media:", round(d["mean_conf"], 3))
print("-" * 60)
print(d["text_ordered"][:1500])

**Revisa el texto de arriba.** ¿Se lee como un CV, con cargos y fechas seguidos? Si sí, continúa.
Si el orden se ve raro, avísame antes de lanzar el OCR completo.

## 8. Descarga COMPLETA (800 por ocupación = 4000 CVs)

In [ ]:
%cd /content/HW_04_202602_cv_experience_ocr_bert
N_PER_OCC = 800   # usa 400 si no te alcanza el tiempo (mínimo permitido: 2000 en total)
# En segundo plano (nohup): no depende de que la celda siga abierta. Buffer pequeño = poca RAM.
!nohup python -u src/download_data.py --n-per-occ {N_PER_OCC} --seed 42 --buffer-size 500 > download.log 2>&1 &

In [ ]:
# Ejecuta de vez en cuando hasta ver "Imágenes guardadas: 4000" (unos 10-15 min)
!tail -n 4 download.log
!pgrep -af download_data.py | grep -v pgrep || echo "TERMINÓ / NO CORRE"

## 9. OCR COMPLETO
Es lo más lento. Si Colab se corta, vuelve a ejecutar las celdas 2 a 6 y luego esta celda:
**retoma solo las imágenes que faltan** (no repite las que ya tienen JSON).

In [ ]:
# En segundo plano; retoma solo las imágenes que faltan si se relanza.
!nohup python -u src/ocr_extraction.py > ocr.log 2>&1 &

In [ ]:
# Ejecuta cada 5-10 min. Termina con "CVs procesados" y "Fallidos: 0" (unos 35 min para 4000 CVs)
!tail -c 300 ocr.log
!pgrep -af ocr_extraction.py | grep -v pgrep || echo "TERMINÓ / NO CORRE"

## 10. Verificación final

In [ ]:
import glob, pandas as pd
n_img = len(glob.glob("data/raw/images/*.png"))
n_ocr = len(glob.glob("data/ocr/*.json"))
print(f"Imágenes: {n_img} | JSON de OCR: {n_ocr}")
meta = pd.read_csv("data/raw/metadata.csv")
print(meta.groupby("occupation").size())
print(meta[["exp_year", "sex", "race"]].isna().sum())